In [5]:
"""
make_test_suite.py  --  bigger golden suite + Python-side statistics for the RTL simulation.

Reads ONLY the exported design (config.json + .mem files) and the raw TEST data; it does not use the training state,
so it is also an independent re-implementation of the integer pipeline.

Test shots written to  <design_dir>/golden_big/  (same file formats as golden/):
    1. N_REAL        real test shots (first N_REAL shots of the test file, with true labels)
    2. N_RANDOM_FULL random full-scale int16 shots  (accumulator / normalizer saturation, sign handling)
    3. N_RANDOM_ADC  random ADC-like Gaussian noise shots
    4. 4             corner shots (+max, -max, zero, alternating)
Usage:  python make_test_suite.py design_freeze/single
"""
import os
import sys
import json
import numpy as np

DESIGN_DIR = "design_freeze/single"
DATA_DIR = r"D:\MSU\Qubit_readout_FPGA\python"
N_REAL = 2000
N_RANDOM_FULL = 100
N_RANDOM_ADC = 100
SEED = 7


def read_mem(path):
    out = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("//"):
                out.append(int(line, 16))
    return out


def signed(x, bits):
    x &= (1 << bits) - 1
    return x - (1 << bits) if x >> (bits - 1) else x


def hexw(x, bits):
    return format(int(x) & ((1 << bits) - 1), "0{}x".format((bits + 3) // 4))


def find_file(fname):
    for d in [DATA_DIR, os.path.join(DATA_DIR, "python"), "."]:
        cand = os.path.join(d, fname)
        if os.path.exists(cand):
            return cand
    raise FileNotFoundError(fname)


cfg = json.load(open(os.path.join(DESIGN_DIR, "config.json")))
assert cfg["n_qubits"] == 1, "this suite is for the single-qubit design"
T = cfg["T_bins"]
wd = cfg["widths"]
IN, DLO, F, WB, MW = wd["input_bits"], wd["dlo_bits"], wd["F_bits"], wd["w_bits"], wd["mem_word"]
lin = cfg["variants"]["lin"]

# ------------------------------------------------------------------ load the exported design from the .mem files
A = np.zeros((4, T), np.int64)
B = np.zeros((4, T), np.int64)
for t, v in enumerate(read_mem(os.path.join(DESIGN_DIR, "dlo_q0.mem"))):
    for k in range(4):
        A[k, t] = signed(v >> ((2 * k) * DLO), DLO)
        B[k, t] = signed(v >> ((2 * k + 1) * DLO), DLO)
mu = np.array([signed(x, MW) for x in read_mem(os.path.join(DESIGN_DIR, "norm_mu_q0.mem"))], np.int64)
n = np.array(read_mem(os.path.join(DESIGN_DIR, "norm_n_q0.mem")), np.int64)
w = np.array([signed(x, WB) for x in read_mem(os.path.join(DESIGN_DIR, "lin", "cls_w_q0.mem"))], np.int64)
thr = signed(read_mem(os.path.join(DESIGN_DIR, "lin", "cls_thr_q0.mem"))[0], MW)


def run(I, Q):
    acc = np.stack([I @ A[k] + Q @ B[k] for k in range(4)], 1)
    one = np.int64(1)
    tmp = np.clip(acc + (one << n) - mu, 0, one << (n + 1))
    v = ((tmp << F) >> (n + 1)) - (one << (F - 1))
    O = np.concatenate([np.full((len(v), 1), 1 << F, np.int64), v], 1)
    score = O @ w
    return acc, v, score, (score > thr).astype(np.int64)


# ------------------------------------------------------------------ build the shots
rng = np.random.default_rng(SEED)
X = np.load(find_file("0528_X_test_0_770.npy"), mmap_mode="r")
y = np.load(find_file("0528_y_test_0_770.npy")).astype(int)
# NOTE: the test file is sorted by label (its first shots are all one class), so pick the real shots at random.
real_idx = np.sort(rng.choice(len(y), size=N_REAL, replace=False))
x = np.asarray(X[real_idx, :2 * T])
assert np.abs(x - np.rint(x)).max() == 0, "test data is not integer ADC counts"
I_real, Q_real = np.rint(x[:, 0::2]).astype(np.int64), np.rint(x[:, 1::2]).astype(np.int64)
lab_real = y[real_idx]
assert (lab_real == 0).any() and (lab_real == 1).any(), "the real shots must contain both classes"

mx = 2 ** (IN - 1) - 1
I_full = rng.integers(-2 ** (IN - 1), mx + 1, size=(N_RANDOM_FULL, T))
Q_full = rng.integers(-2 ** (IN - 1), mx + 1, size=(N_RANDOM_FULL, T))
I_adc = np.clip(np.rint(rng.normal(0, 1000, (N_RANDOM_ADC, T))), -4883, 4883).astype(np.int64)
Q_adc = np.clip(np.rint(rng.normal(0, 1000, (N_RANDOM_ADC, T))), -4883, 4883).astype(np.int64)
alt = np.where(np.arange(T) % 2 == 0, mx, -mx)
I_cor = np.stack([np.full(T, mx), np.full(T, -mx), np.zeros(T, int), alt]).astype(np.int64)
Q_cor = np.stack([np.full(T, -mx), np.full(T, mx), np.zeros(T, int), -alt]).astype(np.int64)

I_all = np.concatenate([I_real, I_full, I_adc, I_cor])
Q_all = np.concatenate([Q_real, Q_full, Q_adc, Q_cor])
labels = np.concatenate([lab_real, np.full(len(I_all) - N_REAL, 2)])
acc, v, score, pred = run(I_all, Q_all)
N_TOTAL = len(I_all)

# ------------------------------------------------------------------ write the testbench files
out = os.path.join(DESIGN_DIR, "golden_big")
os.makedirs(out, exist_ok=True)


def write(name, words, comment):
    with open(os.path.join(out, name), "w") as f:
        f.write("// " + comment + "\n" + "\n".join(words) + "\n")


write("tb_input.mem", [format(((int(Q_all[i, t]) & 0xFFFF) << 16) | (int(I_all[i, t]) & 0xFFFF), "08x")
                       for i in range(N_TOTAL) for t in range(T)], "inputs {Q16,I16}, T words per shot")
write("tb_label.mem", [hexw(l, 4) for l in labels], "true label (0/1) for real shots, 2 = no label")
write("tb_acc.mem", [hexw(acc[i, k], MW) for i in range(N_TOTAL) for k in range(4)], "expected accumulators")
write("tb_v.mem", [hexw(v[i, k], 16) for i in range(N_TOTAL) for k in range(4)], "expected normalized features")
write("tb_score_lin.mem", [hexw(s, MW) for s in score], "expected scores")
write("tb_pred_lin.mem", [hexw(p, 1) for p in pred], "expected decisions")

# ------------------------------------------------------------------ Python-side statistics
sat_v = (np.abs(v) == 2 ** (F - 1)).any(axis=1)
real = slice(0, N_REAL)
p_real = pred[real]
f0 = np.mean(p_real[lab_real == 0] == 0)
f1 = np.mean(p_real[lab_real == 1] == 1)
bits = lambda a: int(np.ceil(np.log2(np.abs(a).max() + 1))) + 1
stats = dict(
    N_TOTAL=int(N_TOTAL), N_REAL=int(N_REAL), n_random_full=N_RANDOM_FULL, n_random_adc=N_RANDOM_ADC,
    model_fidelity_real=dict(F0=float(f0), F1=float(f1), F=float(0.5 * (f0 + f1))),
    acc_bits_used_real=bits(acc[real]), acc_bits_used_all=bits(acc), acc_bits_worst_case=wd["acc_bits_worst_case"],
    score_bits_used_real=bits(score[real]), score_bits_used_all=bits(score), score_bits_declared=lin["score_bits"],
    frac_real_shots_saturating_v=float(sat_v[real].mean()),
    frac_random_full_saturating_v=float(sat_v[N_REAL:N_REAL + N_RANDOM_FULL].mean()),
    frac_random_adc_saturating_v=float(sat_v[N_REAL + N_RANDOM_FULL:N_REAL + N_RANDOM_FULL + N_RANDOM_ADC].mean()),
)
json.dump(stats, open(os.path.join(out, "summary.json"), "w"), indent=2)

print(f"wrote {out}  ({N_TOTAL} shots: {N_REAL} real + {N_RANDOM_FULL} random full-scale + {N_RANDOM_ADC} ADC-like + 4 corner)")
print(f"real shots per class: {np.bincount(lab_real).tolist()}")
print(f"python model on the {N_REAL} real shots: F0={f0:.4f} F1={f1:.4f} F={0.5 * (f0 + f1):.4f}  "
      f"(standard error about +/-{100 * np.sqrt(0.04 * 0.96 / N_REAL):.2f} points: small sample, NOT a performance claim)")
print(f"accumulator bits actually used: real {stats['acc_bits_used_real']}, all {stats['acc_bits_used_all']} "
      f"(hardware width {wd['acc_bits_worst_case']})")
print(f"score bits actually used: real {stats['score_bits_used_real']}, all {stats['score_bits_used_all']} "
      f"(declared {lin['score_bits']})")
print(f"shots whose v hits the clip limit: real {100 * stats['frac_real_shots_saturating_v']:.2f}%, "
      f"random full-scale {100 * stats['frac_random_full_saturating_v']:.0f}%, ADC-like {100 * stats['frac_random_adc_saturating_v']:.0f}%")
print(f"\nTestbench parameters:  N_TOTAL={N_TOTAL}  N_REAL={N_REAL}  SCORE_BITS={lin['score_bits']}")

wrote design_freeze/single\golden_big  (2204 shots: 2000 real + 100 random full-scale + 100 ADC-like + 4 corner)
real shots per class: [1015, 985]
python model on the 2000 real shots: F0=0.9704 F1=0.9513 F=0.9609  (standard error about +/-0.44 points: small sample, NOT a performance claim)
accumulator bits actually used: real 28, all 34 (hardware width 34)
score bits actually used: real 23, all 23 (declared 23)
shots whose v hits the clip limit: real 0.00%, random full-scale 98%, ADC-like 0%

Testbench parameters:  N_TOTAL=2204  N_REAL=2000  SCORE_BITS=23
